# Crime data downloader

Data sacada 
https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2/about_data

https://data.lacity.org/Public-Safety/Crime-Data-from-2020-to-Present/2nrs-mtv8/about_data

In [3]:
import requests
import os
import csv
import time

## Configuraciones

In [4]:
# SET OFFSET
download_Chicago_url = CHICAGO_API_URL + '?$offset=1000'

# save path
save_path='./data/'

# Police stations data download apis
overpass_url = "https://overpass-api.de/api/interpreter"
# "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
# "https://overpass.kumi.systems/api/interpreter"


years = [2020, 2021, 2022, 2023, 2024, 2025]

# Chicago
CHICAGO_API_URL='https://data.cityofchicago.org/resource/ijzp-q8t2.csv'
chicago_save_path = os.path.join(save_path, "chicago")
chicago_name = 'chicago'
chicago_bbox = (41.64, -87.52, 42.02, -87.52)  # (min_lat, min_lon, max_lat, max_lon)

# New York
NEW_YORK_API_URL= 'https://data.cityofnewyork.us/resource/5uac-w243.csv'
newyork_save_path = os.path.join(save_path, "new_york")
newyork_name = 'new_york'
newyork_bbox = (40.49, -74.26, 40.92, -87.82)



## Police Station data download

In [18]:
def download_police_stations(city_name, city_save_path, bbox):
    min_lat, min_lon, max_lat, max_lon = bbox
    
    # Query Overpass
    overpass_query = f"""
    [out:json][timeout:90];
    (
      node["amenity"="police"]({min_lat},{min_lon},{max_lat},{max_lon});
      way["amenity"="police"]({min_lat},{min_lon},{max_lat},{max_lon});
      relation["amenity"="police"]({min_lat},{min_lon},{max_lat},{max_lon});
    );
    out center;
    """
    
    data = None
    
    try:
        print(f"Consultando Overpass API ({city_name}): {overpass_url}")
        response = requests.get(overpass_url, params={"data": overpass_query}, timeout=100)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"Fallo en {overpass_url}: {e}")
    
    if not data or "elements" not in data:
        raise RuntimeError(f"No se pudo obtener data de Overpass para {city_name}")
    
    output_dir = os.path.join(city_save_path, "police_stations")
    os.makedirs(output_dir, exist_ok=True)
    
    file_path = os.path.join(output_dir, f"{city_name}_police_stations.csv")
    
    with open(file_path, mode="w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["id", "type", "name", "lat", "lon"])
        
        for el in data["elements"]:
            lat = el.get("lat") or el.get("center", {}).get("lat")
            lon = el.get("lon") or el.get("center", {}).get("lon")
            name = el.get("tags", {}).get("name", "")
            writer.writerow([el.get("id"), el.get("type"), name, lat, lon])
    
    print(f"Datos descargados: {len(data['elements'])} estaciones -> {file_path}")
    return file_path

print("="*60)
print("Descargando estaciones de policía...")
print("="*60)

download_police_stations(chicago_name, chicago_save_path, chicago_bbox)
print()
download_police_stations(newyork_name, newyork_save_path, newyork_bbox)
print()

Descargando estaciones de policía...
Consultando Overpass API (chicago): https://overpass-api.de/api/interpreter
Datos descargados: 0 estaciones -> ./data/chicago\police_stations\chicago_police_stations.csv

Consultando Overpass API (new_york): https://overpass-api.de/api/interpreter
Datos descargados: 1799 estaciones -> ./data/new_york\police_stations\new_york_police_stations.csv



In [6]:
def download_transport_stations(city_name, city_save_path, bbox):
    min_lat, min_lon, max_lat, max_lon = bbox
    
    # Query Overpass for bus stations, train stations, and subway entrances
    overpass_query = f"""
    [out:json][timeout:90];
    (
      node["amenity"="bus_station"]({min_lat},{min_lon},{max_lat},{max_lon});
      way["amenity"="bus_station"]({min_lat},{min_lon},{max_lat},{max_lon});
      relation["amenity"="bus_station"]({min_lat},{min_lon},{max_lat},{max_lon});
      node["railway"="station"]({min_lat},{min_lon},{max_lat},{max_lon});
      way["railway"="station"]({min_lat},{min_lon},{max_lat},{max_lon});
      relation["railway"="station"]({min_lat},{min_lon},{max_lat},{max_lon});
      node["railway"="subway_entrance"]({min_lat},{min_lon},{max_lat},{max_lon});
      node["public_transport"="station"]({min_lat},{min_lon},{max_lat},{max_lon});
      way["public_transport"="station"]({min_lat},{min_lon},{max_lat},{max_lon});
    );
    out center;
    """
    
    data = None
    
    try:
        print(f"Consultando Overpass API para transporte público ({city_name}): {overpass_url}")
        response = requests.get(overpass_url, params={"data": overpass_query}, timeout=100)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        print(f"Fallo en {overpass_url}: {e}")
    
    if not data or "elements" not in data:
        raise RuntimeError(f"No se pudo obtener data de transporte público de Overpass para {city_name}")
    
    output_dir = os.path.join(city_save_path, "transport_stations")
    os.makedirs(output_dir, exist_ok=True)
    
    file_path = os.path.join(output_dir, f"{city_name}_transport_stations.csv")
    
    with open(file_path, mode="w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["id", "type", "transport_type", "name", "lat", "lon"])
        
        for el in data["elements"]:
            lat = el.get("lat") or el.get("center", {}).get("lat")
            lon = el.get("lon") or el.get("center", {}).get("lon")
            tags = el.get("tags", {})
            name = tags.get("name", "")
            
            # Determine transport type
            if tags.get("amenity") == "bus_station":
                transport_type = "bus_station"
            elif tags.get("railway") == "station":
                transport_type = "train_station"
            elif tags.get("railway") == "subway_entrance":
                transport_type = "subway_entrance"
            elif tags.get("public_transport") == "station":
                transport_type = "public_transport_station"
            else:
                transport_type = "unknown"
            
            writer.writerow([el.get("id"), el.get("type"), transport_type, name, lat, lon])
    
    print(f"Datos descargados: {len(data['elements'])} estaciones de transporte -> {file_path}")
    return file_path

print("="*60)
print("Descargando estaciones de transporte público...")
print("="*60)

download_transport_stations(chicago_name, chicago_save_path, chicago_bbox)
print()
download_transport_stations(newyork_name, newyork_save_path, newyork_bbox)
print()

Descargando estaciones de transporte público...
Consultando Overpass API para transporte público (chicago): https://overpass-api.de/api/interpreter
Datos descargados: 0 estaciones de transporte -> ./data/chicago\transport_stations\chicago_transport_stations.csv

Consultando Overpass API para transporte público (new_york): https://overpass-api.de/api/interpreter
Datos descargados: 5912 estaciones de transporte -> ./data/new_york\transport_stations\new_york_transport_stations.csv



## Public Transport Stations data download

## Chicago crime data download

In [9]:
#Chicago
for i in range(2020, 2025):
    download_year(download_Chicago_url, year=i, save_path=save_path,name='chicago_crime_offset1000')

Downloaded data for year 2020 to ./data
Downloaded data for year 2021 to ./data
Downloaded data for year 2022 to ./data
Downloaded data for year 2023 to ./data
Downloaded data for year 2024 to ./data


In [19]:
#LA
LA_API_URL ='https://data.lacity.org/resource/2nrs-mtv8.csv'
download_LA_url = LA_API_URL + '?$offset=1000'

In [20]:
def download_year_la(url, year, save_path, name):
    """Download LA crime data filtered by date_occ column (floating_timestamp)"""
    # Build date range filter for floating_timestamp column
    start_date = f"{year}-01-01T00:00:00.000"
    end_date = f"{year}-12-31T23:59:59.999"
    filter_param = f"$where=date_occ between '{start_date}' and '{end_date}'"
    
    # Add filter to the URL
    if '?' in url:
        filtered_url = f"{url}&{filter_param}"
    else:
        filtered_url = f"{url}?{filter_param}"
    
    # Download the file
    response = requests.get(filtered_url)
    response.raise_for_status()  # Check for errors
    
    # Save to file
    filename = f"{save_path}/{name}_{year}.csv"
    with open(filename, 'wb') as f:
        f.write(response.content)
    
    print(f"Downloaded LA data for year {year} to {filename}")
    return filename

In [22]:
#LA - Download data for years 2020-2025 using date_occ filter
for i in range(2020, 2026):
    download_year_la(download_LA_url, year=i, save_path=save_path, name='la_crime_offset1000')

Downloaded LA data for year 2025 to ./data/la_crime_offset1000_2025.csv
